# 实验六：视频为什么会卡、糊、停？
## Network Impairments and Video Playback

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：~12–15 分钟  
**运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **GPU 非必需** &nbsp;|&nbsp; **Internet Off**

---

### 一句话问题

> **同样是“网络变差”，为什么有时视频会糊、有时会卡、有时会冻结？**

本实验不再把 Packet Loss、Jitter、Bandwidth Drop 直接画成三种静态图像滤镜，而是把它们放回真正的**时间轴、接收缓冲和播放器行为**中。


## Demo2 / Demo3 → Demo6：把网络、质量与播放真正串起来

此前我们已经分别学习：

- **Demo2**：Bandwidth → ABR → QoE
- **Demo3**：Visual Quality → PSNR / SSIM

Demo6 把它们第一次合到完整链路里：

**Network → Receiver → Buffer → Playback → User QoE**

本实验重点观察：

- Packet Loss
- Burst Loss
- Jitter
- Jitter Buffer
- Bandwidth Drop
- Rebuffer / Freeze
- Quality Switch
- Playback Delay


## 学习目标

完成本实验后，你应该能够：

1. 解释 Packet Loss 为什么不等于“固定出现灰色块”；
2. 区分 **Random Loss** 与 **Burst Loss**；
3. 理解 Jitter 是**到达时间变化**，不是空间图像错位；
4. 理解 Jitter Buffer 如何在 **Latency 与 Stability** 之间取舍；
5. 解释 Bandwidth Drop 为什么可能导致 Freeze，也可能导致 ABR 降低画质；
6. 区分 Network KPI 与 Playback QoE；
7. 读懂统一 Playback Dashboard；
8. 理解 ABR、buffering、concealment、retransmission/FEC 在系统中的不同位置。


## 运行环境

| 项目 | 设置 |
|---|---|
| 平台 | Kaggle Notebook |
| 主计算资源 | CPU |
| GPU | 不需要；可作为后续 AI concealment 扩展 |
| Internet | Off |
| 外部数据 | 不需要 |
| 视频 | 由 `skimage.data.astronaut()` 离线生成 6 秒运动序列 |
| 帧率 | 10 fps |
| 分辨率 | 320×180 |
| 依赖 | NumPy、Pandas、Matplotlib、Pillow、scikit-image |

直接 **Run All** 即可。


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display, Image as IPImage

from skimage import data, img_as_float32
from skimage.transform import resize
from skimage.metrics import structural_similarity

SEED = 2026
rng = np.random.default_rng(SEED)

FPS = 10
DURATION = 6
N_FRAMES = FPS * DURATION
HEIGHT = 180
WIDTH = 320
FRAME_INTERVAL_MS = 1000 / FPS

print("Environment ready")
print(f"Video: {DURATION}s | {FPS} fps | {N_FRAMES} frames | {WIDTH}x{HEIGHT}")


# 第一幕：先生成一个真正有时间维度的短视频

视频不是“一张图片经过网络”。

它是：

**Frame 0 → Frame 1 → Frame 2 → … → Frame 59**

我们用内置自然图像生成一个 6 秒、10 fps 的运动序列：

- 背景做缓慢水平移动；
- 叠加一个移动目标；
- 加入帧号和时间戳。

这样后续的 Freeze、Late Frame 和 Quality Switch 都能在**时间维度**上真正观察。


In [2]:
base = img_as_float32(data.astronaut())
base = resize(base, (HEIGHT, WIDTH), anti_aliasing=True, preserve_range=True).astype(np.float32)

frames = []
for i in range(N_FRAMES):
    shift = int(round(10 * np.sin(2*np.pi*i/N_FRAMES)))
    frame = np.roll(base, shift=shift, axis=1).copy()

    # Add a moving square target.
    x = int(20 + (WIDTH - 60) * i / (N_FRAMES - 1))
    y = int(120 + 20 * np.sin(i / 7))
    frame[max(0,y-12):min(HEIGHT,y+12), max(0,x-12):min(WIDTH,x+12), :] = [1.0, 0.2, 0.2]

    # Timestamp overlay using PIL.
    pil = Image.fromarray(np.uint8(np.clip(frame*255, 0, 255)))
    draw = ImageDraw.Draw(pil)
    draw.rectangle((4, 4, 120, 24), fill=(0,0,0))
    draw.text((8, 7), f"t={i/FPS:0.1f}s  F{i:02d}", fill=(255,255,255))
    frames.append(np.asarray(pil).astype(np.float32)/255.0)

frames = np.stack(frames)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, idx in zip(axes, [0, 15, 30, 45]):
    ax.imshow(frames[idx])
    ax.set_title(f"Frame {idx} | {idx/FPS:.1f}s")
    ax.axis("off")
plt.suptitle("Clean Motion Sequence")
plt.tight_layout()
plt.show()


# 第二幕：Packet Loss —— 同样 10%，为什么体验不同？

一帧视频通常会跨越多个网络包。这里用教学模型把每帧拆成 **12 个 packets**。

我们构造两种网络：

### Random Loss
丢包均匀分散。

### Burst Loss
相同总丢包数，但更集中地连续发生。

为了模拟“帧是否还能及时正确解码”，定义：

> 一帧中丢失 ≥ 4 个 packets → 该帧视为不可用。

Receiver 使用最简单的 concealment：

> **重复上一帧。**

重点不是模拟某个具体 codec，而是观察：

> **平均 Loss Rate 相同，并不代表 Freeze Pattern 相同。**


In [3]:
PACKETS_PER_FRAME = 12
LOSS_RATE = 0.10
TOTAL_PACKETS = N_FRAMES * PACKETS_PER_FRAME
N_LOST = int(round(TOTAL_PACKETS * LOSS_RATE))

# Random loss: same total packet count.
random_loss_flat = np.zeros(TOTAL_PACKETS, dtype=bool)
random_idx = rng.choice(TOTAL_PACKETS, size=N_LOST, replace=False)
random_loss_flat[random_idx] = True
random_loss = random_loss_flat.reshape(N_FRAMES, PACKETS_PER_FRAME)

# Burst loss: same total packet count, concentrated in contiguous runs.
burst_loss_flat = np.zeros(TOTAL_PACKETS, dtype=bool)
remaining = N_LOST
starts = [8*PACKETS_PER_FRAME, 25*PACKETS_PER_FRAME, 41*PACKETS_PER_FRAME]
cursor = 0
for s in starts:
    if remaining <= 0:
        break
    run = min(remaining, 24)  # about two complete frames per burst
    burst_loss_flat[s:s+run] = True
    remaining -= run
# if any packets remain, place them in one last burst
if remaining > 0:
    s = 52*PACKETS_PER_FRAME
    burst_loss_flat[s:s+remaining] = True
burst_loss = burst_loss_flat.reshape(N_FRAMES, PACKETS_PER_FRAME)

def frame_failures(packet_loss):
    lost_per_frame = packet_loss.sum(axis=1)
    failed = lost_per_frame >= 4
    return lost_per_frame, failed

rand_lost_pf, rand_failed = frame_failures(random_loss)
burst_lost_pf, burst_failed = frame_failures(burst_loss)

def longest_run(mask):
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best

loss_summary = pd.DataFrame([
    {
        "Pattern": "Random",
        "Packet Loss Rate": random_loss.mean(),
        "Failed Frames": int(rand_failed.sum()),
        "Longest Freeze (frames)": longest_run(rand_failed),
        "Longest Freeze (s)": longest_run(rand_failed)/FPS,
    },
    {
        "Pattern": "Burst",
        "Packet Loss Rate": burst_loss.mean(),
        "Failed Frames": int(burst_failed.sum()),
        "Longest Freeze (frames)": longest_run(burst_failed),
        "Longest Freeze (s)": longest_run(burst_failed)/FPS,
    },
])

display(loss_summary.style.format({
    "Packet Loss Rate": "{:.1%}",
    "Longest Freeze (s)": "{:.2f}",
}))

plt.figure(figsize=(12,4))
plt.step(np.arange(N_FRAMES)/FPS, rand_lost_pf, where="mid", label="Random loss")
plt.step(np.arange(N_FRAMES)/FPS, burst_lost_pf, where="mid", label="Burst loss")
plt.axhline(4, linestyle="--", label="Frame failure threshold")
plt.xlabel("Video time (s)")
plt.ylabel("Lost packets in frame")
plt.title("Same Packet Loss Rate, Different Temporal Pattern")
plt.grid(alpha=0.2)
plt.legend()
plt.show()


## Concealment：丢帧以后播放器显示什么？

本实验不再把 Packet Loss 固定画成“灰块”。

如果某帧不可用，我们采用最简单策略：

> **Repeat Previous Frame**

于是网络层面的丢包，最终表现为：

**Packet Loss → Frame unavailable → Concealment → Freeze**

真实系统还可能使用：

- retransmission / NACK
- FEC
- codec error resilience
- decoder concealment

所以“丢包长什么样”并不存在唯一答案。


In [4]:
def conceal_with_previous(video, failed_mask):
    out = video.copy()
    last_good = out[0].copy()
    for i in range(len(out)):
        if failed_mask[i] and i > 0:
            out[i] = last_good
        else:
            last_good = out[i].copy()
    return out

random_playback = conceal_with_previous(frames, rand_failed)
burst_playback = conceal_with_previous(frames, burst_failed)

fig, axes = plt.subplots(2, 4, figsize=(14,6))
sample_idx = [8, 9, 25, 26]
for c, idx in enumerate(sample_idx):
    axes[0,c].imshow(random_playback[idx])
    axes[0,c].set_title(f"Random | F{idx}")
    axes[0,c].axis("off")

    axes[1,c].imshow(burst_playback[idx])
    axes[1,c].set_title(f"Burst | F{idx}")
    axes[1,c].axis("off")

plt.suptitle("Loss Concealment by Repeating the Previous Frame")
plt.tight_layout()
plt.show()


# 第三幕：Jitter 是“时间问题”，不是“画面横向错位”

正常情况下，每隔 100 ms 发送一帧。

但网络引入的 delay variation 会使实际到达时间变得不规则：

**Send Time ≠ Arrival Time**

这才是 Jitter 的核心。

下面生成一条带有基础延迟、随机波动和几个 delay spike 的 arrival timeline。


In [5]:
send_ms = np.arange(N_FRAMES) * FRAME_INTERVAL_MS

base_delay_ms = 60.0
jitter_noise = rng.normal(0, 25, N_FRAMES)

# Add three delay spikes.
spikes = np.zeros(N_FRAMES)
spikes[15:18] += [70, 140, 90]
spikes[34:37] += [120, 180, 80]
spikes[49:51] += [160, 100]

network_delay_ms = np.clip(base_delay_ms + jitter_noise + spikes, 5, None)
arrival_ms = send_ms + network_delay_ms

jitter_std_ms = np.std(network_delay_ms - np.mean(network_delay_ms))

plt.figure(figsize=(12,4))
plt.plot(send_ms/1000, network_delay_ms, marker="o")
plt.axhline(base_delay_ms, linestyle="--", label="Base delay")
plt.xlabel("Frame send time (s)")
plt.ylabel("Network delay (ms)")
plt.title(f"Frame Delay Variation | std ≈ {jitter_std_ms:.1f} ms")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.scatter(send_ms/1000, send_ms/1000, label="Ideal arrival (relative)")
plt.scatter(send_ms/1000, arrival_ms/1000, label="Actual arrival")
plt.xlabel("Send time (s)")
plt.ylabel("Absolute time (s)")
plt.title("Send Timeline vs Arrival Timeline")
plt.grid(alpha=0.2)
plt.legend()
plt.show()


# 第四幕：Jitter Buffer Challenge

播放器可以故意增加 playout delay，让晚到的帧有更多时间赶上。

本实验比较：

- **0 ms buffer**
- **100 ms buffer**
- **200 ms buffer**

简化规则：

> deadline = send time + base playout delay + jitter buffer

如果 frame 到达时间超过 deadline，就算 **Late Frame**，播放端需要冻结或跳过。

因此：

> **更大的 Jitter Buffer → 更少 Late Frames，但更高端到端延迟。**


In [6]:
BASE_PLAYOUT_MS = 80
JITTER_BUFFERS = [0, 100, 200]

jitter_rows = []
jitter_masks = {}

for jb in JITTER_BUFFERS:
    deadline = send_ms + BASE_PLAYOUT_MS + jb
    late = arrival_ms > deadline
    jitter_masks[jb] = late

    jitter_rows.append({
        "Jitter Buffer (ms)": jb,
        "Late Frames": int(late.sum()),
        "Late Frame Rate": late.mean(),
        "Freeze Time (s)": late.sum()/FPS,
        "Added Delay (ms)": jb,
    })

jitter_table = pd.DataFrame(jitter_rows)
display(jitter_table.style.format({
    "Late Frame Rate": "{:.1%}",
    "Freeze Time (s)": "{:.2f}",
}))

plt.figure(figsize=(8,5))
plt.plot(
    jitter_table["Added Delay (ms)"],
    jitter_table["Late Frames"],
    marker="o"
)
plt.xlabel("Added Jitter Buffer Delay (ms)")
plt.ylabel("Late Frames")
plt.title("Latency vs Playback Stability")
plt.grid(alpha=0.2)
plt.show()


# 第五幕：Bandwidth Drop —— “卡”还是“糊”，取决于播放器策略

带宽下降本身并不等于“画面变模糊”。

如果播放器坚持高码率：

> **Bandwidth ↓ → Download slower → Buffer ↓ → Rebuffer / Freeze**

如果使用 ABR：

> **Bandwidth ↓ → Lower bitrate / resolution → Quality ↓ but playback continues**

这正好与 Demo2 的 ABR 概念衔接。


In [7]:
SEGMENT_DURATION = 0.5
N_SEGMENTS = int(DURATION / SEGMENT_DURATION)
segment_time = np.arange(N_SEGMENTS) * SEGMENT_DURATION

bandwidth = np.array([
    10.0, 9.5, 9.0, 8.5,   # 0-2s
    3.0, 2.5, 2.2, 2.8,    # 2-4s
    6.0, 6.5, 7.0, 7.5     # 4-6s
])

BITRATES = np.array([2.0, 4.0, 8.0])  # Mbps

def simulate_player(bandwidth, policy, initial_buffer=1.5):
    buffer = initial_buffer
    prev_action = 1
    rows = []

    for i, bw in enumerate(bandwidth):
        action = policy(buffer, bw, prev_action)
        bitrate = BITRATES[action]

        download_time = bitrate * SEGMENT_DURATION / bw
        rebuffer = max(download_time - buffer, 0.0)
        new_buffer = max(buffer - download_time, 0.0) + SEGMENT_DURATION

        rows.append({
            "segment": i,
            "time": i * SEGMENT_DURATION,
            "bandwidth": bw,
            "bitrate": bitrate,
            "buffer_before": buffer,
            "download_time": download_time,
            "rebuffer": rebuffer,
            "buffer_after": new_buffer,
        })

        buffer = new_buffer
        prev_action = action

    return pd.DataFrame(rows)

def fixed_policy(buffer, bw, prev):
    return 2  # 8 Mbps

def abr_policy(buffer, bw, prev):
    # Conservative throughput + buffer heuristic.
    if buffer < 0.7 or bw < 3.2:
        return 0
    if buffer < 1.2 or bw < 6.0:
        return 1
    return 2

fixed_df = simulate_player(bandwidth, fixed_policy)
abr_df = simulate_player(bandwidth, abr_policy)

player_summary = pd.DataFrame([
    {
        "Strategy": "Fixed 8 Mbps",
        "Avg Bitrate": fixed_df["bitrate"].mean(),
        "Rebuffer (s)": fixed_df["rebuffer"].sum(),
        "Min Buffer (s)": fixed_df["buffer_after"].min(),
    },
    {
        "Strategy": "ABR",
        "Avg Bitrate": abr_df["bitrate"].mean(),
        "Rebuffer (s)": abr_df["rebuffer"].sum(),
        "Min Buffer (s)": abr_df["buffer_after"].min(),
    }
])

display(player_summary.style.format({
    "Avg Bitrate": "{:.2f}",
    "Rebuffer (s)": "{:.2f}",
    "Min Buffer (s)": "{:.2f}",
}))

plt.figure(figsize=(12,4))
plt.step(segment_time, bandwidth, where="post", linewidth=2, label="Bandwidth")
plt.step(segment_time, fixed_df["bitrate"], where="post", label="Fixed bitrate")
plt.step(segment_time, abr_df["bitrate"], where="post", label="ABR bitrate")
plt.xlabel("Video time (s)")
plt.ylabel("Mbps")
plt.title("Bandwidth Drop: Fixed Bitrate vs ABR")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.plot(segment_time, fixed_df["buffer_after"], marker="o", label="Fixed buffer")
plt.plot(segment_time, abr_df["buffer_after"], marker="o", label="ABR buffer")
plt.xlabel("Video time (s)")
plt.ylabel("Buffer (s)")
plt.title("Player Buffer Evolution")
plt.grid(alpha=0.2)
plt.legend()
plt.show()


# 第六幕：同一时刻，“高清冻结”和“低清播放”哪个体验更好？

为了把系统结果变成直观画面：

- Fixed bitrate 在发生 rebuffer 时：**重复上一帧**；
- ABR 在低码率段：**降低空间细节，但继续播放**。

因此用户可能面对：

> **High Quality but Frozen**  
> vs  
> **Lower Quality but Playing**

这比“Bandwidth Drop = Blur”更接近真实播放器行为。


In [8]:
def lower_quality(frame, bitrate):
    if bitrate >= 8:
        return frame
    if bitrate >= 4:
        small = resize(frame, (HEIGHT//2, WIDTH//2), anti_aliasing=True, preserve_range=True)
    else:
        small = resize(frame, (HEIGHT//4, WIDTH//4), anti_aliasing=True, preserve_range=True)
    return resize(small, (HEIGHT, WIDTH), order=1, anti_aliasing=False, preserve_range=True).astype(np.float32)

# Map segment decisions to frame-level display.
fixed_playback = frames.copy()
abr_playback = frames.copy()

freeze_until_frame = -1
for seg in range(N_SEGMENTS):
    start = int(seg * SEGMENT_DURATION * FPS)
    end = int((seg+1) * SEGMENT_DURATION * FPS)

    # ABR quality adaptation.
    br = float(abr_df.loc[seg, "bitrate"])
    for f in range(start, min(end, N_FRAMES)):
        abr_playback[f] = lower_quality(frames[f], br)

    # Fixed: approximate rebuffer by repeating previous visible frame.
    reb = float(fixed_df.loc[seg, "rebuffer"])
    freeze_frames = int(round(reb * FPS))
    if freeze_frames > 0:
        freeze_start = start
        freeze_end = min(N_FRAMES, freeze_start + freeze_frames)
        ref_idx = max(0, freeze_start - 1)
        fixed_playback[freeze_start:freeze_end] = fixed_playback[ref_idx]

# Pick a frame during low-bandwidth period.
idx = 30
fig, axes = plt.subplots(1,3,figsize=(13,4))
axes[0].imshow(frames[idx]); axes[0].set_title("Clean Reference")
axes[1].imshow(fixed_playback[idx]); axes[1].set_title("Fixed 8 Mbps\nMay Freeze")
axes[2].imshow(abr_playback[idx]); axes[2].set_title("ABR\nLower Quality, Keeps Playing")
for ax in axes: ax.axis("off")
plt.suptitle(f"Same Moment: t={idx/FPS:.1f}s")
plt.tight_layout()
plt.show()


# 第七幕：Playback Dashboard —— 网络指标怎样传导成用户体验？

统一时间轴展示：

1. Bandwidth
2. Packet Loss / Burst
3. Arrival Delay / Jitter
4. Player Buffer
5. Freeze / Late Frame events

这张图是 Demo6 的系统级核心。

> **Network KPI ≠ Playback QoE**

例如：

- 相同 10% Loss Rate，Random 与 Burst 的最长 Freeze 不同；
- 更大的 Jitter Buffer 减少 Late Frames，却增加延迟；
- 相同 Bandwidth Drop，Fixed bitrate 可能卡顿，而 ABR 可能降低质量继续播放。


In [9]:
fig = plt.figure(figsize=(13,12))

ax1 = plt.subplot(5,1,1)
ax1.step(segment_time, bandwidth, where="post", linewidth=2)
ax1.set_ylabel("Mbps")
ax1.set_title("Playback Dashboard")
ax1.grid(alpha=0.2)

ax2 = plt.subplot(5,1,2, sharex=ax1)
ax2.vlines(np.where(burst_failed)[0]/FPS, 0, 1, linewidth=3)
ax2.set_ylabel("Burst\nloss frames")
ax2.set_ylim(0,1.2)
ax2.grid(alpha=0.2)

ax3 = plt.subplot(5,1,3, sharex=ax1)
ax3.plot(send_ms/1000, network_delay_ms)
ax3.set_ylabel("Delay\n(ms)")
ax3.grid(alpha=0.2)

ax4 = plt.subplot(5,1,4, sharex=ax1)
ax4.plot(segment_time, fixed_df["buffer_after"], marker="o", label="Fixed")
ax4.plot(segment_time, abr_df["buffer_after"], marker="o", label="ABR")
ax4.set_ylabel("Buffer\n(s)")
ax4.legend()
ax4.grid(alpha=0.2)

ax5 = plt.subplot(5,1,5, sharex=ax1)
late100 = jitter_masks[100]
ax5.vlines(np.where(late100)[0]/FPS, 0, 1, label="Late frame @100ms JB")
ax5.set_ylabel("Playback\nevents")
ax5.set_xlabel("Video time (s)")
ax5.set_ylim(0,1.2)
ax5.legend()
ax5.grid(alpha=0.2)

plt.tight_layout()
plt.show()


# 第八幕：做一个 6 秒并排播放 GIF

下面把三种播放体验并排：

- **Clean**
- **Impaired**：Burst Loss concealment + Fixed bitrate freeze
- **Mitigated**：ABR quality adaptation + 100 ms jitter buffer 思路

GIF 只是教学展示，不是实际 codec/network emulator。

它的作用是把前面的时间轴指标变成可以直接观看的结果。


In [10]:
# Build a compact side-by-side animated GIF.
gif_path = Path("/tmp/demo6_playback_comparison.gif")

impaired = burst_playback.copy()

# Inject fixed-bitrate freeze segments into impaired sequence.
for seg in range(N_SEGMENTS):
    reb = float(fixed_df.loc[seg, "rebuffer"])
    freeze_frames = int(round(reb * FPS))
    if freeze_frames > 0:
        start = int(seg * SEGMENT_DURATION * FPS)
        end = min(N_FRAMES, start + freeze_frames)
        ref_idx = max(0, start - 1)
        impaired[start:end] = impaired[ref_idx]

mitigated = abr_playback.copy()

# Also conceal late frames after a 100 ms jitter buffer by repeating prior frame.
late100 = jitter_masks[100]
mitigated = conceal_with_previous(mitigated, late100)

def labeled_panel(frame, label):
    pil = Image.fromarray(np.uint8(np.clip(frame*255,0,255)))
    draw = ImageDraw.Draw(pil)
    draw.rectangle((0,0,WIDTH,22), fill=(0,0,0))
    draw.text((8,5), label, fill=(255,255,255))
    return pil

gif_frames = []
for i in range(N_FRAMES):
    panels = [
        labeled_panel(frames[i], "Clean"),
        labeled_panel(impaired[i], "Impaired"),
        labeled_panel(mitigated[i], "Mitigated"),
    ]
    canvas = Image.new("RGB", (WIDTH*3, HEIGHT), "black")
    for j,p in enumerate(panels):
        canvas.paste(p, (j*WIDTH,0))
    gif_frames.append(canvas)

gif_frames[0].save(
    gif_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=int(1000/FPS),
    loop=0,
    optimize=True,
)

print(f"GIF created: {gif_path} | size={gif_path.stat().st_size/1024:.1f} KiB")
display(IPImage(filename=str(gif_path)))


# 真实系统里这些机制分别放在哪里？

### Packet Loss
可能通过：

- retransmission / NACK
- FEC
- codec error resilience
- decoder concealment

缓解。

### Jitter
主要是：

- packet/frame arrival variation
- jitter buffer
- playout scheduling

问题。

### Bandwidth Drop
主要影响：

- segment / frame 是否能及时下载
- buffer level
- ABR bitrate / resolution selection
- rebuffering risk

因此，不应把三者简单映射成固定的“灰块 / 撕裂 / 模糊”。

更准确的系统视角是：

> **网络损伤发生在传输链路，但用户最终感知的是 Freeze、Delay、Rebuffer 与 Quality Switch。**


## 可选 GPU 扩展：AI Frame Concealment

主实验故意不依赖 GPU，因为本 Demo 的核心是：

**Network Timeline → Receiver → Buffer → Playback**

如果后续希望增加 AI 扩展，可以在丢帧位置比较：

1. Repeat Previous Frame
2. Linear / optical-flow interpolation
3. Neural frame interpolation / concealment

例如 RIFE、BasicVSR 类模型可能需要 GPU、模型权重和额外依赖，因此不放入主流程，以保持：

> **Run All、Internet Off、可重复、课堂轻量**

如果课程进入“AI for Media Delivery”专题，再单独增加 GPU 版扩展更合适。


## 实验局限性与思考

### 有意简化
1. 视频由静态自然图像合成运动，不是实际编码后的 MP4；
2. Packetization 与 frame failure threshold 是教学模型；
3. 没有模拟 GOP、I/P/B frames；
4. 没有真实 RTP/QUIC/TCP 协议栈；
5. Jitter Buffer 使用简化 deadline 模型；
6. ABR 使用教学版 heuristic；
7. 没有真实 NACK/FEC；
8. GIF 只用于教学视觉化，不代表真实播放器实现。

### 思考题
1. 为什么相同 Packet Loss Rate 下，Burst Loss 往往更伤观看体验？
2. Jitter Buffer 为什么不能无限增大？
3. 低 Bandwidth 时，为什么“降低画质继续播放”有时优于“保持高清等待”？
4. 如果加入 FEC，Packet Loss、Bandwidth 和 Latency 之间会出现什么新权衡？
5. I-frame 丢失与普通 P-frame 丢失的影响为什么可能不同？
6. Demo2 的 ABR 策略如何直接接入本 Demo？
7. Demo3 的 PSNR / SSIM 应该在哪一层使用？
8. 怎样把本实验扩展成真实 WebRTC Stats Dashboard？


---

← [实验五：语义通信 vs 传统传输](https://www.kaggle.com/code/guopingtan/fmi-demo5-semantic-communication)
&nbsp;|&nbsp;
🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)
&nbsp;|&nbsp;
[实验七：网络损伤对 VR/360° 视频的影响 →](https://www.kaggle.com/code/guopingtan/fmi-demo-7-network-impairments-on-vr-360-video)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University
